# MERIT — Layer 2: credit assignment

Layer 1 produced the ledger. Layer 2 turns it into a number per entry:
**utility, with a confidence interval and a trial count.**

The model, one observation per turn:

```
y_t = a + sum_i beta_i * z_it + sum_i g_i * elig_it + controls_t + e_t
```

`elig_it` = entry *i* was a retrieval candidate on turn *t*.
`z_it` = it was a candidate **and** survived masking.

Both columns matter. Randomisation only happened among candidates — if an entry
was never retrieved, no coin was flipped for it. Conditioning on `elig` restores
the experiment, so `beta_i` is the effect of injecting entry *i* given it was
retrievable.

No model calls anywhere in this notebook. It is a regression over logs.

In [1]:
print("running")

running


In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))

import pandas as pd
pd.set_option("display.width", 160)

from merit import build_store
from merit import credit as C

store = build_store()
planted = {eid: e.planted for eid, e in store.entries.items()}

## 1. Load the ledger

Point this at whichever run you want to analyse. Use the live run for the real
result; the mock run is for checking the estimator itself.

In [3]:
DB  = "data/merit.db"
RUN = "live_400"

led = C.load_ledger(DB, RUN)
n_turns = led["turn_id"].nunique()
checks = {
    "rows present":         len(led) > 0,
    "400 turns":            n_turns == 400,
    "all rows settled":     (led["settled"] == 1).all(),
    "masked rows exist":    (led["masked"] == 1).sum() > 0,
    "empty rate < 5%":      led["outcome_score"].isna().mean() < 0.05,
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'BAD'}  {k}")
print(f"\n{len(led)} rows, {n_turns} turns")

OK   rows present
OK   400 turns
OK   all rows settled
OK   masked rows exist
OK   empty rate < 5%

1600 rows, 400 turns


## 2. Fit

`min_trials` drops entries that were never injected — or never withheld — often
enough to be identified. Estimating those anyway produces a number with no
experiment behind it, which is worse than reporting nothing.

In [4]:
utility = C.fit_utility(led, min_trials=5)
utility

[design] dropping 3 under-sampled entries: e_misc_2, e_misc_3, e_misc_4
[fit] 389 turns, 7 entries, 16 design columns
[fit] R^2 = 0.685


,entry_id,utility,se,ci_lo,ci_hi,p_value,n_injected,n_withheld,promotable,demotable
0,e_deploy_flag,1.0464,0.0311,0.9856,1.1073,0.00000,94,16,True,False
1,e_test_flag,0.4208,0.0616,0.3001,0.5416,0.00000,273,53,True,False
2,e_build_flag,0.3713,0.0724,0.2294,0.5131,0.00000,284,41,True,False
3,e_build_noise,0.0656,0.0486,-0.0297,0.1609,0.17747,222,42,False,False
4,e_deploy_noise,0.0382,0.0380,-0.0363,0.1126,0.31487,127,29,False,False
5,e_test_noise,0.0308,0.0355,-0.0388,0.1003,0.38588,240,37,False,False
6,e_misc_1,-0.0068,0.0520,-0.1087,0.0951,0.89630,37,11,False,False


Read the columns:

- **utility** — change in P(task succeeds) from injecting this entry
- **ci_lo / ci_hi** — 95% interval, HC3 robust
- **promotable** — `ci_lo > 0`. This is Layer 4's promote gate
- **demotable** — `ci_hi < 0`. The entry measurably hurts

## 3. Did it recover the planted structure?

The labels were hidden from retrieval, masking, the model and the detector. If
the estimator assigns positive utility to `useful` and zero to `distractor`,
it is measuring effect rather than retrievability.

In [5]:
C.recovery_report(utility, store)

,planted,entries,mean_utility,min_utility,max_utility,n_promotable
0,distractor,3,0.045,0.031,0.066,0
1,irrelevant,1,-0.007,-0.007,-0.007,0
2,useful,3,0.613,0.371,1.046,3


In [6]:
print(C.verdict(utility, store))

PASS: all 3 useful entries are promotable (CI excludes zero) and no distractor or irrelevant entry is. Utility-gated promotion works. Proceed to Layer 3.


## 4. The headline: utility vs recall frequency

This is the comparison the paper rests on.

`recall_rank` is what the incumbent promotion rule ranks by — how often an entry
gets retrieved. OpenClaw-style consolidation scores candidates on retrieval
relevance, recall frequency, query diversity and recurrence. Every one of those
measures *retrievability*.

`rank_shift` is `recall_rank - utility_rank`. A large positive shift is an entry
the incumbent rule under-values. A large negative shift is one it would promote
that does nothing.

In [7]:
comparison = C.compare_rankings(utility, C.recall_frequency(led), planted)
comparison

,entry_id,planted,utility,ci_lo,ci_hi,promotable,retrievals,recall_rank,utility_rank,rank_shift
0,e_deploy_flag,useful,1.0464,0.9856,1.1073,True,112,6,1,5
1,e_test_flag,useful,0.4208,0.3001,0.5416,True,335,2,2,0
2,e_build_flag,useful,0.3713,0.2294,0.5131,True,336,1,3,-2
3,e_build_noise,distractor,0.0656,-0.0297,0.1609,False,270,4,4,0
4,e_deploy_noise,distractor,0.0382,-0.0363,0.1126,False,162,5,5,0
5,e_test_noise,distractor,0.0308,-0.0388,0.1003,False,286,3,6,-3
6,e_misc_1,irrelevant,-0.0068,-0.1087,0.0951,False,49,7,7,0


In [8]:
top_recall = comparison.nsmallest(3, "recall_rank")[["entry_id","planted","utility","promotable"]]
top_utility = comparison.nsmallest(3, "utility_rank")[["entry_id","planted","utility","promotable"]]
print("Promoted by recall frequency:"); display(top_recall)
print("Promoted by measured utility:"); display(top_utility)

Promoted by recall frequency:


,entry_id,planted,utility,promotable
2,e_build_flag,useful,0.3713,True
1,e_test_flag,useful,0.4208,True
5,e_test_noise,distractor,0.0308,False


Promoted by measured utility:


,entry_id,planted,utility,promotable
0,e_deploy_flag,useful,1.0464,True
1,e_test_flag,useful,0.4208,True
2,e_build_flag,useful,0.3713,True


## 5. Power: how many turns do you need?

Refit on growing prefixes. The row where every useful entry is promotable and
`false_promotions` is 0 is the sample size to report in the paper.

`false_promotions` is the number that matters most — a distractor clearing the
gate means Layer 4 would promote dead weight into the always-injected core.

In [9]:
C.power_curve(led, store, sizes=(100, 200, 300, 400))

[design] dropping 5 under-sampled entries: e_deploy_flag, e_misc_1, e_misc_2, e_misc_3, e_misc_4
[design] dropping 4 under-sampled entries: e_misc_1, e_misc_2, e_misc_3, e_misc_4
[design] dropping 3 under-sampled entries: e_misc_2, e_misc_3, e_misc_4
[design] dropping 3 under-sampled entries: e_misc_2, e_misc_3, e_misc_4


,turns,fitted,entries_fitted,useful_promotable,false_promotions,mean_useful,mean_other,mean_ci_width
0,100,True,5,1/2,0,0.184,0.045,0.460
1,200,True,6,3/3,0,0.531,0.071,0.246
2,300,True,7,3/3,0,0.629,0.041,0.223
3,400,True,7,3/3,0,0.613,0.032,0.190


## 6. Robustness: logistic link

The outcome is binary and the main fit is a linear probability model, which can
predict outside [0,1]. Logit average marginal effects should give a similar
ordering. If they disagree sharply, the LPM is being pushed too hard.

In [12]:
try:
    logit = C.fit_logit(led, min_trials=5)
    display(logit.merge(utility[["entry_id","utility"]], on="entry_id"))
except Exception as exc:
    print("logit did not converge (common on small or separable samples):", exc)

[design] dropping 3 under-sampled entries: e_misc_2, e_misc_3, e_misc_4


/Users/sadi_/miniconda3/envs/merit/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


,entry_id,utility_logit_ame,se,utility
0,e_deploy_flag,2.9896,548.9575,1.0464
1,e_test_flag,1.2130,250.6479,0.4208
2,e_build_flag,1.2090,250.6479,0.3713
3,e_deploy_noise,0.5713,221.6473,0.0382
4,e_test_noise,0.0381,0.0405,0.0308
5,e_build_noise,0.0270,0.0228,0.0656
6,e_misc_1,-0.1055,1163.6568,-0.0068


## 7. Control ablation

Does the task-family control actually matter? Fit with and without it.

Layer 1's Phase 0 check showed that pooling across families dilutes real effects
toward zero, because retrieval mixes families into one candidate list and an
entry accumulates 'absent' observations on turns where it was irrelevant.
This quantifies that.

In [13]:
with_ctrl = C.fit_utility(led, verbose=False)[["entry_id","utility","ci_lo","promotable"]]
no_family  = C.fit_utility(led, controls=("n_candidates","bundle_tokens"),
                           verbose=False)[["entry_id","utility","promotable"]]
ab = with_ctrl.merge(no_family, on="entry_id", suffixes=("_full","_nofam"))
ab["planted"] = ab["entry_id"].map(planted)
ab["dilution"] = (ab["utility_full"] - ab["utility_nofam"]).round(3)
ab

[design] dropping 3 under-sampled entries: e_misc_2, e_misc_3, e_misc_4
[design] dropping 3 under-sampled entries: e_misc_2, e_misc_3, e_misc_4


,entry_id,utility_full,ci_lo,promotable_full,utility_nofam,promotable_nofam,planted,dilution
0,e_deploy_flag,1.0464,0.9856,True,1.0812,True,useful,-0.035
1,e_test_flag,0.4208,0.3001,True,0.4461,True,useful,-0.025
2,e_build_flag,0.3713,0.2294,True,0.3725,True,useful,-0.001
3,e_build_noise,0.0656,-0.0297,False,0.0603,False,distractor,0.005
4,e_deploy_noise,0.0382,-0.0363,False,0.0793,False,distractor,-0.041
5,e_test_noise,0.0308,-0.0388,False,0.0237,False,distractor,0.007
6,e_misc_1,-0.0068,-0.1087,False,-0.0392,False,irrelevant,0.032


## 8. Save

The utility table is Layer 3's input — it is what retrieval will rank by.

In [14]:
from pathlib import Path
OUT = Path("../data"); OUT.mkdir(exist_ok=True)
utility.to_csv(OUT / f"utility_{RUN}.csv", index=False)
comparison.to_csv(OUT / f"ranking_comparison_{RUN}.csv", index=False)
print("saved utility and ranking comparison")

saved utility and ranking comparison
